# Lenguajes y Autómatas II (SCD-1016)
## Sesión 19 · Estructuras (struct) y Offsets de Miembros

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Dónde queda cada campo de una estructura dentro de su bloque de memoria?
>
> **Objetivo de la sesión:** Calcular los offsets de los miembros de una estructura aplicando alineación y padding, y obtener el tamaño total del tipo.

**Entregable:** tus funciones offset_campo() y tamano_struct() en artesano_sesion19_c.c, la salida del harness y la bitácora en auditorias/auditoria_sesion19.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes una lista ordenada de campos con su tipo; offset_campo() devuelve la posición del campo pedido y tamano_struct() el total con padding final. El harness compara contra la disposición real de la arquitectura.

Al final de la sesión tendrás: **Un calculador de offsets y tamaños de struct que respeta alineación y padding, verificado con cuatro disposiciones distintas.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 19](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion19.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion19.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion19.h"

int offset_campo(const Campo *campos, int total, const char *nombre);
int tamano_struct(const Campo *campos, int total);
```

- Cada campo se alinea a su propio ancho antes de asignarle offset.
- El tamaño total se redondea al múltiplo del campo más grande.
- Un campo inexistente, una lista NULL o un tipo desconocido devuelven -1.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion19_c.c` junto con `artesano_sesion19_c.c`
> (o `ia_sesion19_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `struct char e int: b` | 4 |
| 2 | `struct de tres campos: c` | 8 |
| 3 | `tamano total con padding` | 12 |
| 4 | `double seguido de char` | 16 |
| 5 | `miembro inexistente` | -1 |
| 6 | `struct sin campos` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 19 - Offsets de miembros y tamano de struct ===
Caso                               esperado:              obtenido:              resultado
[ 1] struct char e int: b               esperado: 4            obtenido: 4            OK
[ 2] struct de tres campos: c           esperado: 8            obtenido: 8            OK
[ 3] tamano total con padding           esperado: 12           obtenido: 12           OK
[ 4] double seguido de char             esperado: 16           obtenido: 16           OK
[ 5] miembro inexistente                esperado: -1           obtenido: -1           OK
[ 6] struct sin campos                  esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion19.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Cada offset coincide con la posición real del campo después de aplicar el relleno.
- [ ] El tamaño total incluye el padding final hasta el múltiplo de la alineación mayor.
- [ ] Un miembro inexistente devuelve -1 y una estructura sin campos devuelve 0.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion19_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion19.h
/* api_sesion19.h - contrato de la Sesion 19
 * Offsets de miembros y tamaño de struct
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION19_H
#define API_SESION19_H

typedef struct Campo {
    const char *nombre;
    const char *tipo;
} Campo;

int offset_campo(const Campo *campos, int total, const char *nombre);
int tamano_struct(const Campo *campos, int total);

#endif /* API_SESION19_H */


Writing api_sesion19.h


In [3]:
%%writefile artesano_sesion19_c.c
/* artesano_sesion19_c.c - Fase 1: El Artesano (sin IA)
 * Offsets de miembros y tamaño de struct.
 *
 * Implementa lo que declara api_sesion19.h.
 * El harness compila este archivo junto con el suyo: no escribas main().
 */
#include "api_sesion19.h"
#include <stdio.h>
#include <string.h>

static int ancho(const char *tipo) {
    if (tipo == NULL) return -1;
    if (strcmp(tipo, "char") == 0)    return 1;
    if (strcmp(tipo, "short") == 0)   return 2;
    if (strcmp(tipo, "int") == 0)     return 4;
    if (strcmp(tipo, "float") == 0)   return 4;
    if (strcmp(tipo, "long") == 0)    return 8;
    if (strcmp(tipo, "double") == 0)  return 8;
    if (strcmp(tipo, "puntero") == 0) return 8;
    return -1;
}

/* sube 'x' al siguiente multiplo de 'a' */
static int alinear(int x, int a) {
    return (x + a - 1) / a * a;
}

int offset_campo(const Campo *campos, int total, const char *nombre) {
    if (campos == NULL || nombre == NULL || total < 0) return -1;
    int pos = 0;
    for (int i = 0; i < total; i++) {
        int w = ancho(campos[i].tipo);
        if (w < 0) return -1;                    /* tipo desconocido */
        pos = alinear(pos, w);                   /* padding antes del campo */
        if (campos[i].nombre != NULL && strcmp(campos[i].nombre, nombre) == 0) return pos;
        pos += w;
    }
    return -1;                                   /* no existe */
}

int tamano_struct(const Campo *campos, int total) {
    if (campos == NULL || total < 0) return -1;
    int pos = 0, mayor = 1;
    for (int i = 0; i < total; i++) {
        int w = ancho(campos[i].tipo);
        if (w < 0) return -1;
        pos = alinear(pos, w) + w;
        if (w > mayor) mayor = w;
    }
    return (total == 0) ? 0 : alinear(pos, mayor);   /* padding final */
}

Writing artesano_sesion19_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion19_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


In [4]:
%%writefile ia_sesion19_c.c
/* ia_sesion19_c.c - Fase 2: El Cientifico
 *
 * Offsets de miembros y tamano de struct (respuesta de la IA al contrato C-R-E-O).
 */
#include "api_sesion19.h"
#include <stddef.h>
#include <string.h>

static int tamano_tipo(const char *tipo) {
    static const struct { const char *nombre; int ancho; } tabla[] = {
        {"char", 1}, {"short", 2}, {"int", 4}, {"float", 4},
        {"long", 8}, {"double", 8}, {"puntero", 8}
    };
    for (size_t i = 0; i < sizeof(tabla) / sizeof(tabla[0]); i++) {
        if (strcmp(tabla[i].nombre, tipo) == 0) {
            return tabla[i].ancho;
        }
    }
    return -1;
}

int offset_campo(const Campo *campos, int total, const char *nombre) {
    if (campos == NULL || nombre == NULL) {
        return -1;
    }
    int offset = 0;
    for (int i = 0; i < total; i++) {
        int ancho = tamano_tipo(campos[i].tipo);
        if (ancho < 0) {
            return -1;
        }
        offset = (offset + ancho - 1) / ancho * ancho;
        if (strcmp(campos[i].nombre, nombre) == 0) {
            return offset;
        }
        offset += ancho;
    }
    return -1;
}

int tamano_struct(const Campo *campos, int total) {
    if (campos == NULL) {
        return -1;
    }
    int offset = 0;
    int max_alineacion = 1;
    for (int i = 0; i < total; i++) {
        int ancho = tamano_tipo(campos[i].tipo);
        if (ancho < 0) {
            return -1;
        }
        offset = (offset + ancho - 1) / ancho * ancho + ancho;
        if (ancho > max_alineacion) {
            max_alineacion = ancho;
        }
    }
    return (offset + max_alineacion - 1) / max_alineacion * max_alineacion;
}

Writing ia_sesion19_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion19_c.c
/* harness_sesion19_c.c - auditoria automatica de la Sesion 19
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion19_c.c artesano_sesion19_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion19.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 19 - Offsets de miembros y tamano de struct ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { Campo a[] = {{"a", "char"}, {"b", "int"}}; int r = offset_campo(a, 2, "b"); verificar_int("struct char e int: b", (long)(4), (long)(r)); }
    { Campo b[] = {{"a", "char"}, {"b", "int"}, {"c", "short"}}; int r = offset_campo(b, 3, "c"); verificar_int("struct de tres campos: c", (long)(8), (long)(r)); }
    { Campo b[] = {{"a", "char"}, {"b", "int"}, {"c", "short"}}; int r = tamano_struct(b, 3); verificar_int("tamano total con padding", (long)(12), (long)(r)); }
    { Campo d[] = {{"a", "double"}, {"b", "char"}}; int r = tamano_struct(d, 2); verificar_int("double seguido de char", (long)(16), (long)(r)); }
    { Campo b[] = {{"a", "char"}, {"b", "int"}, {"c", "short"}}; int r = offset_campo(b, 3, "z"); verificar_int("miembro inexistente", (long)(-1), (long)(r)); }
    { Campo b[] = {{"a", "char"}}; int r = tamano_struct(b, 0); verificar_int("struct sin campos", (long)(0), (long)(r)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion19.md" "\n");
    return 0;
}


Writing harness_sesion19_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion19_c.c artesano_sesion19_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 19 - Offsets de miembros y tamano de struct ===
Caso                               esperado:              obtenido:              resultado
[ 1] struct char e int: b               esperado: 4            obtenido: 4            OK
[ 2] struct de tres campos: c           esperado: 8            obtenido: 8            OK
[ 3] tamano total con padding           esperado: 12           obtenido: 12           OK
[ 4] double seguido de char             esperado: 16           obtenido: 16           OK
[ 5] miembro inexistente                esperado: -1           obtenido: -1           OK
[ 6] struct sin campos                  esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion19.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion19_c.c ia_sesion19_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 19 - Offsets de miembros y tamano de struct ===
Caso                               esperado:              obtenido:              resultado
[ 1] struct char e int: b               esperado: 4            obtenido: 4            OK
[ 2] struct de tres campos: c           esperado: 8            obtenido: 8            OK
[ 3] tamano total con padding           esperado: 12           obtenido: 12           OK
[ 4] double seguido de char             esperado: 16           obtenido: 16           OK
[ 5] miembro inexistente                esperado: -1           obtenido: -1           OK
[ 6] struct sin campos                  esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion19.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion19.md` a `auditorias/auditoria_sesion19.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
